# Bangladesh GDP Scenario Visualiser — FY2040-41

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AniDPoka/Bangladesh-GDP-Growth/blob/main/notebooks/Bangladesh_GDP_Scenario_Visualiser_2040.ipynb)

**An interactive implementation of the World Bank Long-Term Growth Model (LTGM v4.3)**, calibrated to real World Bank, IMF and Penn World Table data.

This notebook answers one question: *how large could Bangladesh's economy be by FY2040-41, and what drives the answer?*

**What's inside**
1. Inputs, each with its source
2. The model engine (LTGM Equations 2, 3, 5 and 6) and the TFP residual
3. Validation against an independent Excel implementation
4. Five scenarios to FY2040-41
5. Growth decomposition: what actually drives growth
6. Sensitivity analysis: which input matters most
7. **Interactive explorer:** move the sliders yourself

**How to run:** `Runtime → Run all`. The interactive explorer appears at the bottom.

> **Year convention:** the base year is FY2025-26 ($460B). Each projected year is labelled by the year the fiscal year starts, so `2040` means **FY2040-41**.

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, HTML

# Use the Colab renderer when running in Google Colab
try:
    import google.colab  # noqa: F401
    pio.renderers.default = "colab"
except ImportError:
    pass

pd.options.display.float_format = "{:,.2f}".format

# Palette used across every chart
NAVY, BLUE, LBLUE = "#12315E", "#2E75B6", "#5B9BD5"
COLORS = {
    "BDP2100 Climate": "#C0392B",
    "Pessimistic":     "#D46B08",
    "Base Case":       NAVY,
    "Moderate":        "#2E7D4F",
    "Optimistic":      "#7030A0",
}
print("Setup complete.")

## 1. Inputs and sources

Every parameter traces to a named source. One parameter, the labour share (β), has no Bangladesh-specific estimate available, so it uses the World Bank's own cross-country standard and is flagged.

In [ ]:
INPUTS = {
    "Y0":      (460.0,      "Starting GDP, FY2025-26 (US$ bn)",          "IMF WEO / World Bank WDI"),
    "pop0":    (173.562364, "Population, 2024 (millions)",               "World Bank WDI: SP.POP.TOTL"),
    "IY":      (0.307,      "Investment share of GDP",                   "World Bank WDI: NE.GDI.TOTL.ZS (2024)"),
    "delta":   (0.032,      "Capital depreciation rate (δ)",             "Penn World Table 8.1"),
    "KY0":     (2.78,       "Initial capital-output ratio (K/Y)",        "Penn World Table 8.1"),
    "g_h":     (0.013,      "Human capital growth per worker",           "Penn World Table 8.1"),
    "gN":      (0.00951,    "Population growth",                         "WDI, avg. 2015-2024"),
    "g_omega": (0.00322,    "Working-age share growth",                  "WDI: SP.POP.1564.TO.ZS, avg. 2014-2024"),
    "g_part":  (0.00361,    "Labour-force participation growth",         "WDI: SL.TLF.CACT.ZS, avg. 2014-2024"),
    "beta":    (0.5,        "Labour share (β)  ⚑ not Bangladesh-specific", "Pennings (2020), LTGM cross-country average"),
    "g_A":     (0.00527,    "TFP growth, base case (derived, see §2)",   "Residual solved from the inputs above"),
}

BASE = {k: v[0] for k, v in INPUTS.items()}

PLAIN = {"Y0", "pop0", "KY0", "beta"}   # everything else is a rate, shown as %
pd.DataFrame(
    [(k, d, f"{v:,.2f}" if k in PLAIN else f"{v:.3%}", s) for k, (v, d, s) in INPUTS.items()],
    columns=["Key", "Parameter", "Value", "Source"],
).set_index("Key")

## 2. The model engine

LTGM treats the economy like a factory with three inputs: **capital** (machines, infrastructure), **labour** (the workforce), and **productivity** (TFP, how well the two are combined).

Each year the model updates four equations from Pennings (2020):

| Step | Equation | What it does |
|---|---|---|
| Capital per worker (Eq. 5) | $1+g_k = \dfrac{(1-\delta) + (I/Y)\,/\,(K/Y)}{(1+g_N)(1+g_\%)(1+g_\omega)}$ | New investment minus depreciation, spread across a growing workforce |
| Output per worker (Eq. 2) | $1+g_y = (1+g_A)(1+g_k)^{1-\beta}(1+g_h)^{\beta}$ | Combines productivity, capital deepening and human capital |
| Output per person (Eq. 3) | $1+g_y^{pc} = (1+g_\omega)(1+g_\%)(1+g_y)$ | Adds demographic and participation effects |
| Capital-output ratio (Eq. 6) | $(K/Y)_{t+1} = \dfrac{1+g_k}{1+g_y}(K/Y)_t$ | Tracks diminishing returns year by year |

Because K/Y is updated every year, growth slows naturally as capital accumulates. Nothing is adjusted by hand.

**Where 0.527% TFP comes from:** TFP can't be measured directly. Using Bangladesh's actual decade-average growth (5.93%) and the inputs above, the model is solved backwards: whatever growth capital and labour can't explain is productivity. That residual is 0.527% a year.

In [ ]:
def run_ltgm(end_year=2040, **overrides):
    """Run the LTGM forward from FY2025-26 to `end_year`.

    Any input in BASE can be overridden, e.g. run_ltgm(g_A=0.01).
    Returns one row per year, including each source of growth.
    """
    p = {**BASE, **overrides}
    KY, Y, pop = p["KY0"], p["Y0"], p["pop0"]
    rows = [dict(year=2025, gdp=Y, pop=pop, gdp_pc=Y * 1e3 / pop,
                 g_Y=np.nan, KY=KY)]

    for year in range(2026, end_year + 1):
        workforce = (1 + p["gN"]) * (1 + p["g_part"]) * (1 + p["g_omega"])
        g_k   = ((1 - p["delta"]) + p["IY"] / KY) / workforce - 1                                # Eq. 5
        g_y   = (1 + p["g_A"]) * (1 + g_k) ** (1 - p["beta"]) * (1 + p["g_h"]) ** p["beta"] - 1  # Eq. 2
        g_ypc = (1 + p["g_omega"]) * (1 + p["g_part"]) * (1 + g_y) - 1                          # Eq. 3
        g_Y   = (1 + g_ypc) * (1 + p["gN"]) - 1                                                  # headline GDP
        KY    = (1 + g_k) / (1 + g_y) * KY                                                       # Eq. 6

        Y   *= 1 + g_Y
        pop *= 1 + p["gN"]

        rows.append(dict(
            year=year, gdp=Y, pop=pop, gdp_pc=Y * 1e3 / pop, g_Y=g_Y, KY=KY,
            # Exact log decomposition of headline growth
            c_tfp=np.log(1 + p["g_A"]),
            c_capital=(1 - p["beta"]) * np.log(1 + g_k),
            c_human=p["beta"] * np.log(1 + p["g_h"]),
            c_labour=np.log(workforce),
        ))
    return pd.DataFrame(rows).set_index("year")

run_ltgm().head()

### Deriving the base-case TFP growth

The residual is computed below rather than typed in. Starting from the decade-average headline growth of 5.93% (FY17–FY25), population, working-age and participation growth are stripped out to get growth per worker. Capital deepening and human capital are then removed using Eq. 5 and Eq. 2, and what is left is TFP.

In [ ]:
G_HIST = 0.0593   # real GDP growth, decade average FY17-FY25 (BBS / Bangladesh Bank, WDI)

p = BASE
g_ypc = (1 + G_HIST) / (1 + p["gN"]) - 1                                   # headline -> per capita
g_y   = (1 + g_ypc) / ((1 + p["g_omega"]) * (1 + p["g_part"])) - 1       # per capita -> per worker
g_k   = ((1 - p["delta"]) + p["IY"] / p["KY0"]) / (
         (1 + p["gN"]) * (1 + p["g_part"]) * (1 + p["g_omega"])) - 1     # Eq. 5
g_A_residual = (1 + g_y) / ((1 + g_k) ** (1 - p["beta"]) * (1 + p["g_h"]) ** p["beta"]) - 1  # Eq. 2 solved for g_A

steps = pd.DataFrame({
    "Step": ["Headline GDP growth (data)", "Per-capita growth", "Per-worker growth",
             "Capital-per-worker growth", "TFP growth (residual)"],
    "Value": [f"{v:.2%}" for v in (G_HIST, g_ypc, g_y, g_k, g_A_residual)],
}).set_index("Step")
assert abs(g_A_residual - BASE["g_A"]) < 5e-5, "Residual does not match the base-case g_A"
steps

## 3. Validation against the Excel workbook

Before trusting any output, the Python engine is checked against an independent Excel implementation of the same model. Both are run to FY2066-67, and the results must match.

In [ ]:
EXCEL_2066 = {          # g_A,              Excel result (US$ bn)
    "BDP2100 Climate": (0.00527 - 0.0073, 2105),
    "Pessimistic":     (0.0,              2380),
    "Base Case":       (0.00527,          3281),
    "Moderate":        (0.01,             4390),
    "Optimistic":      (0.02,             8193),
}

check = []
for name, (gA, excel) in EXCEL_2066.items():
    python = run_ltgm(end_year=2066, g_A=gA)["gdp"].iloc[-1]
    check.append((name, f"{gA:.3%}", excel, round(python), round(python) - excel))

validation = pd.DataFrame(check, columns=["Scenario", "TFP growth", "Excel ($bn)", "Python ($bn)", "Difference"])
assert (validation["Difference"].abs() <= 1).all(), "Python engine does not match Excel"
print("✅ Python engine matches the Excel workbook for all five scenarios.\n")
validation.set_index("Scenario")

## 4. Five scenarios to FY2040-41

All five scenarios share **identical** capital, labour, investment and human-capital inputs. **Only TFP growth differs.**

| Scenario | TFP growth | Basis |
|---|---|---|
| BDP2100 Climate | −0.20% | Base case minus the Bangladesh Delta Plan 2100's −0.73pp/yr climate drag |
| Pessimistic | 0.00% | LTGM lower benchmark (Pennings 2020) |
| **Base Case** | **0.53%** | **Solved from real data (§2)** |
| Moderate | 1.00% | LTGM mid benchmark |
| Optimistic | 2.00% | LTGM upper benchmark |

In [ ]:
END_YEAR = 2040
CLIMATE_DRAG = 0.0073

SCENARIOS = {
    "BDP2100 Climate": BASE["g_A"] - CLIMATE_DRAG,
    "Pessimistic":     0.0,
    "Base Case":       BASE["g_A"],
    "Moderate":        0.01,
    "Optimistic":      0.02,
}

results = {name: run_ltgm(END_YEAR, g_A=gA) for name, gA in SCENARIOS.items()}

years_ahead = END_YEAR - 2025
summary = pd.DataFrame({
    name: {
        "TFP growth":                 f"{SCENARIOS[name]:.2%}",
        "GDP FY2040-41 ($bn)":        round(df["gdp"].iloc[-1]),
        "× today's economy":          round(df["gdp"].iloc[-1] / BASE["Y0"], 2),
        "GDP per capita ($)":         round(df["gdp_pc"].iloc[-1]),
        "Avg. annual growth":         f"{(df['gdp'].iloc[-1] / BASE['Y0']) ** (1 / years_ahead) - 1:.2%}",
        "Capital-output ratio, 2040": round(df["KY"].iloc[-1], 2),
    }
    for name, df in results.items()
}).T
summary

In [ ]:
fig = go.Figure()
for name, df in results.items():
    fig.add_trace(go.Scatter(
        x=df.index, y=df["gdp"], name=f"{name} ({SCENARIOS[name]:.2%})", mode="lines",
        line=dict(color=COLORS[name], width=4 if name == "Base Case" else 2.5,
                  dash="dash" if name == "BDP2100 Climate" else "solid"),
        hovertemplate="%{x}: $%{y:,.0f}bn<extra>" + name + "</extra>",
    ))
    fig.add_annotation(x=END_YEAR, y=df["gdp"].iloc[-1], text=f"<b>${df['gdp'].iloc[-1]:,.0f}bn</b>",
                       showarrow=False, xanchor="left", xshift=8, font=dict(color=COLORS[name], size=12))

low, high = results["BDP2100 Climate"]["gdp"].iloc[-1], results["Optimistic"]["gdp"].iloc[-1]
fig.update_layout(
    title=dict(text=f"<b>One unmeasurable input moves FY2040-41 GDP from ${low:,.0f}bn to ${high:,.0f}bn</b>"
                    "<br><sup>Five LTGM scenarios, identical inputs except productivity (TFP) growth</sup>"),
    xaxis_title="Fiscal year (starting)", yaxis_title="Real GDP (US$ bn, 2026 prices)",
    template="simple_white", hovermode="x unified", height=520,
    legend=dict(orientation="h", y=-0.2), margin=dict(r=110),
)
fig.show()

## 5. What actually drives growth?

Headline growth can be split exactly into four sources: **TFP**, **capital deepening**, **human capital**, and **workforce growth**. The chart shows each source's share of total growth from FY2025-26 to FY2040-41.

In [ ]:
SOURCES = {"c_tfp": "Productivity (TFP)", "c_capital": "Capital deepening",
           "c_human": "Human capital", "c_labour": "Workforce growth"}
SOURCE_COLORS = ["#E87722", NAVY, BLUE, LBLUE]

decomp = pd.DataFrame({
    name: {label: df[col].sum() for col, label in SOURCES.items()}
    for name, df in results.items()
}).T
decomp_share = decomp.div(decomp.sum(axis=1), axis=0) * 100

fig = go.Figure()
for (label, color) in zip(SOURCES.values(), SOURCE_COLORS):
    fig.add_trace(go.Bar(
        y=decomp_share.index, x=decomp_share[label], name=label, orientation="h",
        marker_color=color, text=decomp_share[label].round(0).astype(int).astype(str) + "%",
        textposition="inside", insidetextanchor="middle",
    ))
fig.update_layout(
    barmode="relative", template="simple_white", height=380,
    title="<b>Without productivity growth, capital has to carry the economy alone</b>"
          "<br><sup>Share of cumulative growth by source, FY2025-26 to FY2040-41</sup>",
    xaxis_title="Share of total growth (%)", legend=dict(orientation="h", y=-0.25),
)
fig.show()

decomp_share.round(1)

**Reading it:** in the Pessimistic case, TFP contributes nothing and capital deepening has to do most of the work, which is why growth is slowest there. Every step up in TFP shifts the burden away from capital, the input that suffers diminishing returns.

## 6. Sensitivity: which input matters most?

Each input is moved across a plausible range while everything else stays at the base case. The wider the bar, the more that input moves FY2040-41 GDP.

In [ ]:
RANGES = {   # input: (low, high, label)
    "g_A":   (0.000, 0.010, "TFP growth: 0% → 1%"),
    "IY":    (0.26,  0.35,  "Investment share: 26% → 35%"),
    "g_h":   (0.008, 0.018, "Human capital growth: 0.8% → 1.8%"),
    "delta": (0.045, 0.025, "Depreciation: 4.5% → 2.5%"),
    "gN":    (0.007, 0.012, "Population growth: 0.7% → 1.2%"),
    "KY0":   (3.0,   2.5,   "Initial K/Y: 3.0 → 2.5"),
}
base_2040 = results["Base Case"]["gdp"].iloc[-1]

rows = []
for key, (lo, hi, label) in RANGES.items():
    gdp_lo = run_ltgm(END_YEAR, **{key: lo})["gdp"].iloc[-1]
    gdp_hi = run_ltgm(END_YEAR, **{key: hi})["gdp"].iloc[-1]
    rows.append((label, min(gdp_lo, gdp_hi), max(gdp_lo, gdp_hi)))

tornado = pd.DataFrame(rows, columns=["Input", "Low", "High"])
tornado["Range"] = tornado["High"] - tornado["Low"]
tornado = tornado.sort_values("Range")

fig = go.Figure()
fig.add_trace(go.Bar(y=tornado["Input"], x=tornado["Low"] - base_2040, base=base_2040,
                     orientation="h", marker_color=LBLUE, name="Low end"))
fig.add_trace(go.Bar(y=tornado["Input"], x=tornado["High"] - base_2040, base=base_2040,
                     orientation="h", marker_color=NAVY, name="High end"))
fig.add_vline(x=base_2040, line_dash="dot", line_color="grey",
              annotation_text=f"Base case ${base_2040:,.0f}bn", annotation_position="top")
fig.update_layout(
    barmode="overlay", template="simple_white", height=420,
    title="<b>Productivity and investment dominate the outcome</b>"
          "<br><sup>FY2040-41 GDP across a plausible range for each input, others held at base case</sup>",
    xaxis_title="GDP FY2040-41 (US$ bn)", legend=dict(orientation="h", y=-0.2),
)
fig.show()

tornado.sort_values("Range", ascending=False).round(0).set_index("Input")

## 7. Interactive explorer

Drag the sliders and the full model re-runs instantly.

- **Scenario buttons** snap the controls to one of the five official cases
- **TFP slider** is the main control
- **Climate toggle** applies the BDP2100 −0.73pp drag on top of your choice
- **Advanced** sliders let you test investment, depreciation and population growth

In [ ]:
_ref = {name: df["gdp"] for name, df in results.items()}

tfp = widgets.FloatSlider(value=BASE["g_A"] * 100, min=-1.0, max=3.0, step=0.05,
                          description="TFP growth %", readout_format=".2f",
                          continuous_update=False, layout=widgets.Layout(width="480px"),
                          style={"description_width": "120px"})
climate = widgets.Checkbox(value=False, description="Apply BDP2100 climate drag (−0.73pp)")
iy = widgets.FloatSlider(value=BASE["IY"] * 100, min=20, max=40, step=0.5, description="Investment % GDP",
                         continuous_update=False, style={"description_width": "120px"},
                         layout=widgets.Layout(width="480px"))
dep = widgets.FloatSlider(value=BASE["delta"] * 100, min=2.0, max=6.0, step=0.1, description="Depreciation %",
                          continuous_update=False, style={"description_width": "120px"},
                          layout=widgets.Layout(width="480px"))
popg = widgets.FloatSlider(value=BASE["gN"] * 100, min=0.4, max=1.6, step=0.05, description="Population growth %",
                           continuous_update=False, style={"description_width": "120px"},
                           layout=widgets.Layout(width="480px"))
preset = widgets.ToggleButtons(options=list(SCENARIOS.keys()), value="Base Case",
                               description="Scenario:", style={"button_width": "130px"})
reset = widgets.Button(description="Reset to base case", button_style="info", icon="refresh")

def _apply_preset(change):
    name = change["new"]
    climate.value = name == "BDP2100 Climate"
    tfp.value = round((BASE["g_A"] if name == "BDP2100 Climate" else SCENARIOS[name]) * 100, 2)
preset.observe(_apply_preset, names="value")

def _reset(_):
    iy.value, dep.value, popg.value = BASE["IY"] * 100, BASE["delta"] * 100, BASE["gN"] * 100
    climate.value = False
    tfp.value = round(BASE["g_A"] * 100, 2)
    preset.value = "Base Case"
reset.on_click(_reset)

def _where(gdp):
    ends = sorted(((s.iloc[-1], n) for n, s in _ref.items()))
    for v, n in ends:                      # within 1% of a scenario counts as "in line"
        if abs(gdp - v) / v < 0.01: return f"In line with the {n} scenario"
    if gdp < ends[0][0]:  return f"Below every official scenario (lowest: {ends[0][1]})"
    if gdp > ends[-1][0]: return f"Above every official scenario (highest: {ends[-1][1]})"
    for (v1, n1), (v2, n2) in zip(ends, ends[1:]):
        if v1 <= gdp <= v2: return f"Between {n1} and {n2}"

def _card(label, value, sub=""):
    return (f"<div style='flex:1;padding:12px 14px;background:#F3F6FB;border-left:4px solid {NAVY};"
            f"margin-right:8px'><div style='font-size:12px;color:#555'>{label}</div>"
            f"<div style='font-size:24px;font-weight:700;color:{NAVY}'>{value}</div>"
            f"<div style='font-size:11px;color:#777'>{sub}</div></div>")

def update(tfp_pct, climate_on, iy_pct, dep_pct, pop_pct):
    gA = tfp_pct / 100 - (CLIMATE_DRAG if climate_on else 0)
    df = run_ltgm(END_YEAR, g_A=gA, IY=iy_pct / 100, delta=dep_pct / 100, gN=pop_pct / 100)
    last = df.iloc[-1]
    cagr = (last["gdp"] / BASE["Y0"]) ** (1 / years_ahead) - 1

    display(HTML(
        "<div style='display:flex;margin:6px 0 4px'>"
        + _card("GDP, FY2040-41", f"${last['gdp']:,.0f}bn", f"{last['gdp']/BASE['Y0']:.2f}× today's $460bn")
        + _card("GDP per capita", f"${last['gdp_pc']:,.0f}", f"population {last['pop']:.0f}M")
        + _card("Avg. annual growth", f"{cagr:.2%}", "FY2026-27 to FY2040-41")
        + _card("Capital-output ratio", f"{last['KY']:.2f}", "starts at 2.78; rising = diminishing returns")
        + "</div>"
        + f"<div style='margin:6px 0 2px;color:{NAVY};font-weight:600'>📍 {_where(last['gdp'])}"
        + f"  ·  effective TFP growth: {gA:.2%}</div>"
    ))

    fig = go.Figure()
    for name, series in _ref.items():
        fig.add_trace(go.Scatter(x=series.index, y=series, name=name, mode="lines", opacity=0.35,
                                 line=dict(color=COLORS[name], width=1.5,
                                           dash="dash" if name == "BDP2100 Climate" else "solid")))
    fig.add_trace(go.Scatter(x=df.index, y=df["gdp"], name="Your scenario", mode="lines",
                             line=dict(color="#E87722", width=5),
                             hovertemplate="%{x}: $%{y:,.0f}bn<extra>Your scenario</extra>"))
    fig.update_layout(template="simple_white", height=430, hovermode="x unified",
                      title="<b>Your scenario vs. the five official cases</b>",
                      xaxis_title="Fiscal year (starting)", yaxis_title="Real GDP (US$ bn)",
                      yaxis=dict(range=[0, max(1500, df["gdp"].max() * 1.08)]),
                      legend=dict(orientation="h", y=-0.22))
    fig.show()

    contrib = {label: df[col].sum() for col, label in SOURCES.items()}
    total = sum(contrib.values())
    bar = go.Figure()
    for (label, val), color in zip(contrib.items(), SOURCE_COLORS):
        bar.add_trace(go.Bar(x=[val / total * 100], y=["Growth sources"], orientation="h", name=label,
                             marker_color=color, text=f"{val / total * 100:.0f}%", textposition="inside"))
    bar.update_layout(barmode="relative", template="simple_white", height=170, showlegend=True,
                      margin=dict(t=40, b=10), title="<b>Where your growth comes from</b>",
                      xaxis=dict(title="", ticksuffix="%"), yaxis=dict(visible=False),
                      legend=dict(orientation="h", y=-0.6))
    bar.show()

out = widgets.interactive_output(update, {"tfp_pct": tfp, "climate_on": climate,
                                          "iy_pct": iy, "dep_pct": dep, "pop_pct": popg})

advanced = widgets.Accordion(children=[widgets.VBox([iy, dep, popg])])
advanced.set_title(0, "Advanced inputs")
advanced.selected_index = None

display(widgets.VBox([
    widgets.HTML(f"<h3 style='color:{NAVY};margin:0'>Bangladesh GDP Scenario Explorer: FY2040-41</h3>"),
    preset, widgets.HBox([tfp, reset]), climate, advanced, out,
]))

## 8. Limitations

- **Population growth is held constant** at the 2015-2024 average. Bangladesh's fertility is declining, so per-capita figures in later years are likely understated and population overstated.
- **The labour share (β = 0.5)** is the World Bank's cross-country standard, not a Bangladesh-specific estimate.
- **TFP is the residual** from a decade that includes both strong (FY17-19) and crisis (FY23-25) years. A different window gives a different base case, which is why five scenarios are shown rather than one forecast.
- **The climate drag** is applied as a constant reduction in TFP, a simplification of BDP2100's business-as-usual estimate.
- **This is a projection, not a prediction.** Its value is showing which assumptions drive the outcome, not pinning down a single number.

**Companion files** in this repository: `model/bangladesh_ltgm_model.xlsx` (the Excel workbook used in §3) and `slides/` (the presentation of these results).

**Sources:** Pennings, S. (2020) *Long Term Growth Model v4.3*, World Bank · World Bank World Development Indicators · IMF World Economic Outlook · Penn World Table 8.1 · Bangladesh Delta Plan 2100.